In [1]:
# script to generate the vcfs for generating sei predictions

In [1]:
# import packages
import pandas as pd
import os

In [2]:
# open all exon filtered predictions (generated in filter_clinvar_preds_4_predictions.ipynb)
os.chdir('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/clinvar/processed_data/mpac_preds')
all_clinvar = pd.concat([pd.read_csv(i, sep = '\t') for i in os.listdir() if i.endswith('.vcf')])
# convert to VCF for Sei
all_vcf4sei = pd.DataFrame({
    '#CHROM' : all_clinvar['chrom'],
    'POS' : all_clinvar['pos'],
    'ID' : all_clinvar['id'],
    'REF' : all_clinvar['ref'],
    'ALT' : all_clinvar['alt'],
    'QUALITY' : ['.' for i in range(len(all_clinvar))],
    'FILTER' : ['.' for i in range(len(all_clinvar))],
    'INFO' : [i.split(';')[1] for i in all_clinvar['id']]
})

In [3]:
# chunk VCF into 20K row chunks for generating predictions
def save_df_chunks(df, base_filename, chunksize=20000):
    n_chunks = (len(df) - 1) // chunksize + 1
    filenames = []
    
    for i in range(n_chunks):
        start_idx = i * chunksize
        end_idx = min((i + 1) * chunksize, len(df))
        chunk = df.iloc[start_idx:end_idx]
        
        filename = f"{base_filename}_chunk_{i + 1}.vcf"
        chunk.to_csv(filename, index=False, sep = '\t')
        filenames.append(filename)
        print(f"Saved {filename}: rows {start_idx} to {end_idx-1}")
    
    print(f"\nTotal: {n_chunks} files created")
    return filenames

In [4]:
save_df_chunks(all_vcf4sei, '../vcfs4sei/clinvar_20260104_sei')

Saved ../vcfs4sei/clinvar_20260104_sei_chunk_1.vcf: rows 0 to 19999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_2.vcf: rows 20000 to 39999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_3.vcf: rows 40000 to 59999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_4.vcf: rows 60000 to 79999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_5.vcf: rows 80000 to 99999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_6.vcf: rows 100000 to 119999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_7.vcf: rows 120000 to 139999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_8.vcf: rows 140000 to 159999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_9.vcf: rows 160000 to 179999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_10.vcf: rows 180000 to 199999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_11.vcf: rows 200000 to 219999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_12.vcf: rows 220000 to 239999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_13.vcf: rows 240000 to 259999
Saved ../vcfs4sei/clinvar_20260104_sei_chunk_14.

['../vcfs4sei/clinvar_20260104_sei_chunk_1.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_2.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_3.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_4.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_5.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_6.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_7.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_8.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_9.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_10.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_11.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_12.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_13.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_14.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_15.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_16.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_17.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_18.vcf',
 '../vcfs4sei/clinvar_20260104_sei_chunk_19.vcf']

In [ ]:
# command for running sei is as follows at:
# /projects/tewhey-lab/buttsj/sei_model/sei-framework
# conda environment: sei_model

#  1 #!/bin/bash
#  2
#  3 #SBATCH --time=1-00:00:00
#  4 #SBATCH --gres=gpu:1
#  5 #SBATCH -n 1
#  6 #SBATCH -p gpu_a100
#  7 #SBATCH -q gpu_training
#  8 #SBATCH --mem 128G
#  9 #SBATCH --mail-type=FAIL,END
# 10 #SBATCH --mail-user=john.butts@jax.org
# 11 #SBATCH --job-name=sei_array_test_20k
# 12 #SBATCH --array=1-17
# 13 #SBATCH --output=clinvar_20260104_slurm_output/slurm_%j_%a.out
# 14
# 15 sh 1_variant_effect_prediction.sh \
# 16 /projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/clinvar/processed_data/vcfs4sei/clinvar_20260104_sei_chunk_${SLURM_ARRAY_TASK_ID}>
# 17 hg38 \
# 18 /projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/clinvar/processed_data/sei_preds \
# 19 --cuda